# Claude CLI Playground
A deliberately buggy notebook for testing Claude Code's debugging, refactoring, and code generation capabilities.

**Bugs seeded across cells** — use Claude CLI to find and fix them.

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from collections import defaultdict
import json
import os

# FIX 1: use a single consistent random seed everywhere
np.random.seed(42)
RANDOM_STATE = 42

## 2. Synthetic Dataset Generation

In [ ]:
def generate_patient_data(n=500):
    """
    Generate a synthetic patient dataset with clinical features.
    Target: readmission within 30 days (binary).
    """
    ages = np.random.normal(60, 15, n).clip(18, 95)
    los = np.random.exponential(scale=5, size=n).clip(1, 30)  # length of stay
    num_comorbidities = np.random.poisson(2.5, n)
    # FIX 2: >= 3 (not > 3) to correctly include patients with exactly 3 comorbidities
    readmitted = ((ages > 65) & (num_comorbidities >= 3) | (los > 10)).astype(int)

    df = pd.DataFrame({
        'age': ages,
        'length_of_stay': los,
        'num_comorbidities': num_comorbidities,
        'num_medications': np.random.randint(1, 20, n),
        'prior_admissions': np.random.poisson(1, n),
        'readmitted_30d': readmitted
    })
    return df

df = generate_patient_data(500)
print(df.shape)
df.head()

## 3. Exploratory Data Analysis

In [ ]:
# FIX 3: df.info() returns None — call df.describe() directly
print(df.describe())

print("\nClass balance:")
print(df['readmitted_30d'].value_counts(normalize=True))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
features = ['age', 'length_of_stay', 'num_comorbidities', 'num_medications', 'prior_admissions']

for i, feat in enumerate(features):
    ax = axes[i // 3, i % 3]
    df.groupby('readmitted_30d')[feat].plot(kind='hist', alpha=0.6, ax=ax, legend=True)
    ax.set_title(feat)
    ax.set_xlabel(feat)

# FIX 4: hide the unused 6th subplot
axes[1, 2].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
feature_cols = ['age', 'length_of_stay', 'num_comorbidities', 'num_medications', 'prior_admissions']
target_col = 'readmitted_30d'

X = df[feature_cols]
y = df[target_col]

# FIX 5: test_size=0.2 gives a more appropriate 400/100 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
# FIX 6: use transform (not fit_transform) on test set to prevent data leakage
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

## 5. Model Training

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)

print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## 6. Feature Importance (Coefficients)

In [ ]:
coef_df = pd.DataFrame({
    'feature': feature_cols,
    # FIX 7: index [0] to get the 1D array from shape (1, n_features)
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)

print(coef_df)

## 7. Custom Metric Tracker

In [ ]:
class ExperimentTracker:
    def __init__(self):
        self.runs = []

    def log(self, name, params, metrics):
        self.runs.append({
            'name': name,
            'params': params,
            'metrics': metrics
        })

    def best_run(self, metric='accuracy'):
        # FIX 8: return just the name and metrics of the best run, not the full dict
        best = max(self.runs, key=lambda r: r['metrics'].get(metric, 0))
        return {'name': best['name'], 'metrics': best['metrics']}

    def to_dataframe(self):
        # FIX 9: flatten nested dicts so params/metrics become individual columns
        return pd.json_normalize(self.runs)


tracker = ExperimentTracker()
tracker.log(
    name='logistic_baseline',
    params={'C': 1.0, 'max_iter': 1000},
    metrics={'accuracy': 0.74, 'f1': 0.71, 'auc': 0.79}
)
tracker.log(
    name='logistic_tuned',
    params={'C': 0.1, 'max_iter': 500},
    metrics={'accuracy': 0.76, 'f1': 0.73, 'auc': 0.81}
)

print("Best run:", tracker.best_run())
print("\nAll runs:")
print(tracker.to_dataframe())

## 8. ICD Code Parser (Bonus Utility)

In [ ]:
def parse_icd_codes(code_string):
    """
    Parse a comma-separated string of ICD-10 codes.
    Returns dict: {code: category}
    """
    icd_map = {
        'E11': 'Type 2 Diabetes',
        'I10': 'Hypertension',
        'J44': 'COPD',
        'N18': 'Chronic Kidney Disease',
        'F32': 'Major Depression',
    }
    # FIX 10: strip() consistently handles any spacing around commas
    codes = [c.strip() for c in code_string.split(',')]
    result = {}
    for code in codes:
        # FIX 11: slice [:3] to correctly capture 3-character ICD prefixes (E11, I10, etc.)
        prefix = code[:3]
        result[code] = icd_map.get(prefix, 'Unknown')
    return result


sample_codes = 'E11.9, I10, J44.1, Z87.891'
print(parse_icd_codes(sample_codes))

## 9. Save Results

In [ ]:
output = {
    'model': 'LogisticRegression',
    'features': feature_cols,
    # FIX 12: save only the positive-class probabilities (column 1), not both columns
    'test_probabilities': y_prob[:, 1].tolist(),
    'predictions': y_pred.tolist()
}

# FIX 13: open file in write mode ('w'), not read mode ('r')
with open('results.json', 'w') as f:
    json.dump(output, f, indent=2)

print("Results saved.")

---
## Bug Index (for reference)

| # | Cell | Bug |
|---|------|-----|
| 1 | Setup | Inconsistent random seed (`np.random.seed(42)` vs `RANDOM_STATE=99`) |
| 2 | Data Gen | Off-by-one: `> 3` should be `>= 3` |
| 3 | EDA | `df.info().describe()` — `info()` returns None |
| 4 | EDA | Empty subplot not hidden with `ax.set_visible(False)` |
| 5 | Preprocessing | `test_size=0.4` too large |
| 6 | Preprocessing | `fit_transform` on test set (data leakage) |
| 7 | Feature Importance | `model.coef_` needs `[0]` index |
| 8 | Tracker | `best_run()` returns full dict instead of useful summary |
| 9 | Tracker | `to_dataframe()` doesn't flatten nested dicts |
| 10 | ICD Parser | Inconsistent delimiter: split on `,` but values have `, ` |
| 11 | ICD Parser | `code[:2]` cuts prefix too short (should be `[:3]`) |
| 12 | Save | `y_prob` is 2D — `.tolist()` works but may not be intended shape |
| 13 | Save | File opened with `'r'` instead of `'w'` |